In [ ]:
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

from baseline_cnn import BaselineCNN
from train_val import train_one_epoch, validate

# 1. Thiết lập đường dẫn theo thư mục dự án
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "clean_dataset.csv"
IMG_DIR = BASE_DIR / "training_images"

# 2. Dataset và chia dữ liệu train/validation
class FashionDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.classes = sorted(self.df["articleType"].dropna().unique())
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.img_dir / f"{row['id']}.jpg"
        with Image.open(image_path) as image:
            image = image.convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.class_to_idx[row["articleType"]]
        return image, label

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu CSV: {CSV_PATH}")
if not IMG_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {IMG_DIR}")

df = pd.read_csv(CSV_PATH).dropna(subset=["id", "articleType"])
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
dataset = FashionDataset(df, IMG_DIR, transform=transform)
labels = df["articleType"].map(dataset.class_to_idx).tolist()
train_indices, val_indices = train_test_split(
    list(range(len(dataset))), test_size=0.2, random_state=42, stratify=labels
)
train_loader = DataLoader(Subset(dataset, train_indices), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(dataset, val_indices), batch_size=32, shuffle=False, num_workers=0)

# 3. Khởi tạo mô hình và huấn luyện
if len(dataset.classes) != 4:
    raise ValueError(f"Mô hình đang cấu hình 4 lớp nhưng dữ liệu có {len(dataset.classes)} lớp: {dataset.classes}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang dùng:", device)
model = BaselineCNN(num_classes=len(dataset.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(5):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model, val_loader, criterion, device)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    print(f"Epoch {epoch + 1}/5 | Train loss: {tr_loss:.4f}, acc: {tr_acc:.4f} | Val loss: {va_loss:.4f}, acc: {va_acc:.4f}")

# 4. Vẽ biểu đồ

epochs = range(1, 6)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs, history["val_loss"], "o-", label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs, history["train_acc"], "o-", label="Train")
axes[1].plot(epochs, history["val_acc"], "o-", label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

# 5. Lưu model trong thư mục dự án
output_path = BASE_DIR / "baseline_cnn.pth"
torch.save(model.state_dict(), output_path)
print(f"Đã lưu model tại: {output_path}")

